# Strength Change Explanations in Quantitative Argumentation
This notebook provides the implementation and evaluation of the the AAMAS-26 paper *Strength Change Explanations in Quantitative Argumentation* by Kampik, Yin, Potyka, and Toni.

We first install and import the required dependencies, and set a random seed.

In [ ]:
! pip3 install numpy tqdm scipy

In [ ]:
import random
import sys
import numpy as np
from tqdm import tqdm
import time
from scipy.stats import kendalltau, spearmanr

sys.path.append("../../src/")
import uncertainpy.gradual as grad

random.seed(1)

Next, we provide code that can generate layered QBAGs with various properties.

In [ ]:
def generate_random_mlp_graph(layer_sizes, connection_prob):
    """
    generate a random MLP-like QBAG structure, represented by DAG

    parameters:
    - layer_sizes: List of integers describing the size of each layer
    - connection_prob: The probability of inter-layer connections (1.0 = fully connected, 0.0 = no connections)

    return:
    - graph: MLP structure in adjacent list (DAG)
    """

    graph = {}  # Store the DAG using an adjacency list
    node_id = 0  # neuron ID
    layer_nodes = []  # Record the neuron IDs in each layer

    # Create neuron nodes
    for size in layer_sizes:
        layer = [node_id + i for i in range(size)]
        layer_nodes.append(layer)
        node_id += size

    # Generate inter-layer connections
    for i in range(len(layer_nodes) - 1):  # Layer-by-layer connection
        for src in layer_nodes[i]:  # Current layer neurons
            for dst in layer_nodes[i+1]:  # Next layer neurons
                if random.uniform(0, 1) < connection_prob:
                    graph.setdefault(src, set()).add(dst)

    # if node not in graph, add empty set
    for layer in layer_nodes:
        for node in layer:
            graph.setdefault(node, set())

    return graph

In [ ]:
def get_layer_nodes(layer_sizes, layer_index=None):
    node_id = 0
    for i, size in enumerate(layer_sizes):
        if i == layer_index:
            return [str(n) for n in range(node_id, node_id + size)]
        node_id += size

    raise ValueError(f"Invalid layer_index {layer_index}: must be between 0 and {len(layer_sizes) - 1}")

In [ ]:
# generate a bespoke MLP-QBAG and output to a file
def generate_bespoke_graph(filename, layer_sizes, connection_prob):
    with open(filename, 'w') as f:
        sys.stdout = f

        # generate the node and edge of a graph
        random_graph = generate_random_mlp_graph(layer_sizes, connection_prob)

        L0 = get_layer_nodes(layer_sizes, len(layer_sizes)-1) # output layer
        L1 = get_layer_nodes(layer_sizes, len(layer_sizes)-2) # last hidden layer
        L2 = get_layer_nodes(layer_sizes, len(layer_sizes)-3)
        L3 = get_layer_nodes(layer_sizes, len(layer_sizes)-4)
        if len(layer_sizes)>=5:
            L4 = get_layer_nodes(layer_sizes, len(layer_sizes)-5)
            L4 = list(map(int, L4))
        L0 = list(map(int, L0))
        L1 = list(map(int, L1))
        L2 = list(map(int, L2))
        L3 = list(map(int, L3))

        # generate base scores for arguments
        for node, edges in random_graph.items():
            # print(node)
            if node in L1:
                random_float = round(random.uniform(0.0, 0.1),2)
            else:
                random_float = round(random.uniform(0.0, 1.0),2)
            print(f"arg({node}, {random_float}).")

        # generate polarity for edges
        for node, edges in random_graph.items():
            for edge in edges:

                if edge in L1:
                    random_boolean = True
                elif edge in L2:
                    random_boolean = False
                else:
                    random_boolean = random.choice([True, False])
                if random_boolean:
                    print(f"att({node}, {edge}).")
                else:
                    print(f"sup({node}, {edge}).")

    sys.stdout = sys.__stdout__

In [ ]:
# generate a random MLP-QBAG and output to a file
def generate_random_graph(filename, layer_sizes, connection_prob):
    with open(filename, 'w') as f:
        sys.stdout = f

        # generate the node and edge of a graph
        random_graph = generate_random_mlp_graph(layer_sizes, connection_prob)

        # generate random base scores for arguments
        for node, edges in random_graph.items():
            random_float = round(random.uniform(0.0, 1.0),2)
            print(f"arg({node}, {random_float}).")

        # generate random polarity for edges
        for node, edges in random_graph.items():
            for edge in edges:
                random_boolean = random.choice([True, False])
                if random_boolean:
                    print(f"att({node}, {edge}).")
                else:
                    print(f"sup({node}, {edge}).")

    sys.stdout = sys.__stdout__

In [ ]:
def generate_qbags(number, layer_sizes, connection_prob=1.0, bespoke=False):
    """
    Generate a random MLP-like QBAG structure, represented by a weighted DAG

    parameters:
    - number: How many QBAG
    - layer_sizes: List of integers describing the size of each layer
    - connection_prob: The probability of inter-layer connections (1.0 = fully connected, 0.0 = no connections)
    - bespoke: Set to `True` if bespoke QBAGs should be generated

    """
    for i in range(N):
        filename = f'../../bags/mlp_{i}.bag'
        if bespoke:
            generate_bespoke_graph(filename, layer_sizes, connection_prob)
        else:
            generate_random_graph(filename, layer_sizes, connection_prob)

We then provide the code for executing our heuristic search for explanations, alongside tools for analyzing the search.

In [ ]:
# obtain input_args, hidden_args, output_args
def get_layer_args(layer_sizes):
    input_size = layer_sizes[0]
    output_size = layer_sizes[-1]
    total_neurons = sum(layer_sizes)

    input_args = [str(i) for i in range(input_size)]

    hidden_start = input_size
    hidden_end = total_neurons - output_size
    hidden_args = [str(i) for i in range(hidden_start, hidden_end)]

    output_args = [str(i) for i in range(hidden_end, total_neurons)]

    return input_args, hidden_args, output_args

In [ ]:
def compute_loss(bag, preferred_order):
    strengths = np.array([bag.arguments[name].strength for name in preferred_order])
    i_idx, j_idx = np.triu_indices(len(strengths), k=1)
    sigma_diff = strengths[j_idx] - strengths[i_idx]
    return np.sum(np.maximum(0,sigma_diff))

In [ ]:
# compute gradient for the loss function
def compute_gradient(h, bag, preferred_order):
    gradient = {}
    original_base_scores = {arg.name: arg.initial_weight for arg in bag.arguments.values()}

    # original penalty
    grad.algorithms.computeStrengthValues(bag, agg_f, inf_f)
    penalty = compute_loss(bag, preferred_order)

    for name, original_weight in original_base_scores.items():
        # perturb current argument
        for arg in bag.arguments.values():
            if arg.name == name:
                arg.reset_initial_weight(original_weight + h)
            else:
                arg.reset_initial_weight(original_base_scores[arg.name])

        grad.algorithms.computeStrengthValues(bag, agg_f, inf_f)
        new_penalty = compute_loss(bag, preferred_order)
        gradient[name] = (new_penalty - penalty) / h

    # restore all base scores
    for arg in bag.arguments.values():
        arg.reset_initial_weight(original_base_scores[arg.name])
    grad.algorithms.computeStrengthValues(bag, agg_f, inf_f)

    return gradient

In [ ]:
# adam optimiser
def adam_gradient(name, gradient, m, v, i):

    grad = gradient[name]
    # Adam optimiser parameters
    learning_rate = 0.1  # Initial learning rate
    beta1 = 0.85            # First-order moment decay rate
    beta2 = 0.98          # Second-order moment decay rate
    epsilon = 1e-8         # a small constant

    # update first-order moment and second-order moment
    m = beta1 * m + (1 - beta1) * grad
    v = beta2 * v + (1 - beta2) * (grad ** 2)

    # bias correction
    m_hat = m / (1 - beta1 ** i)
    v_hat = v / (1 - beta2 ** i)

    update = learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)

    return update, m, v

In [ ]:
# if the order is exactly the same as the desired order, then valid otherwise not.
def is_valid_order(bag, preferred_order):
    strengths = np.array([bag.arguments[name].strength for name in preferred_order])
    return np.all(strengths[:-1] >= strengths[1:])

In [ ]:
# compute kendall and spearman correlations. if decreasing then 1, increasing 0.
def compute_kendall_spearman(predicted_strengths):
    if len(set(predicted_strengths)) == 1:
        return 1.0, 1.0

    ideal_descending = sorted(predicted_strengths, reverse=True)
    kendall_corr, _ = kendalltau(ideal_descending, predicted_strengths)
    spearman_corr, _ = spearmanr(ideal_descending, predicted_strengths)

    return kendall_corr, spearman_corr

Finally, we run the search given several semantics and several sets of synthetically generated QBAGs.

In [ ]:
semantics_list = ['DF-QuAD', 'EB', 'QE']
mutable_args_list = ['bespoke', 'input_mutable', 'hidden_mutable', 'input_hidden_mutable', 'all_mutable']
layer_sizes_list = [[8,32,16,3], [8,32,16,8], [8,64,16,8,3], [8,64,16,8,8]]
N = 100 # number of arguments
h = 10e-6 # for gradient approximation

for semantics in semantics_list:
    for mutable_args in mutable_args_list:
        for layer_sizes in layer_sizes_list:
            # manage layers
            preferred_order = get_layer_nodes(layer_sizes, len(layer_sizes)-1)
            input_args, hidden_args, output_args = get_layer_args(layer_sizes)
            L1 = get_layer_nodes(layer_sizes, len(layer_sizes)-2) # last hidden layer
            
            # generate QBAGs:
            if mutable_args == 'bespoke':
                generate_qbags(N, layer_sizes, 1.0, True)
            else:
                generate_qbags(N, layer_sizes, 1.0, False)
            # specify semantics:
            if semantics == 'DF-QuAD':
                agg_f = grad.semantics.modular.ProductAggregation()
                inf_f = grad.semantics.modular.LinearInfluence(conservativeness=1)
            if semantics == 'EB':
                agg_f = grad.semantics.modular.SumAggregation()
                inf_f = grad.semantics.modular.EulerBasedInfluence()
            if semantics == 'QE':
                agg_f = grad.semantics.modular.SumAggregation()
                inf_f = grad.semantics.modular.QuadraticMaximumInfluence(conservativeness=1)
                
            # specify mutable args:
            if mutable_args == 'bespoke':
                immutable_args = L1
            if mutable_args == 'input_mutable':
                immutable_args = hidden_args + output_args
            if mutable_args == 'hidden_mutable':
                immutable_args = input_args + output_args
            if mutable_args == 'input_hidden_mutable':
                immutable_args = output_args
            if mutable_args == 'all_mutable':
                immutable_args = []
                
            ## run experiments ##
            # compute desired orderings for N MLP-like QBAGs
            valid, kendall, spearman, time_total, base_score_diff = ([0] * N for _ in range(5))
            mutable_num = sum(layer_sizes)-len(immutable_args) # number of mutable arguments
            valid_itself = 0
            
            for i in tqdm(range(N)):
            
                filename = f'../../bags/mlp_{i}.bag'
                bag = grad.BAG(filename)
                if is_valid_order(bag, preferred_order):
                    valid_itself += 1
                    valid[i] = 1
                    kendall[i], spearman[i] = 1, 1
                    continue
            
                start_time = time.time()
                m = {}
                v = {}
                original_base_scores = {arg.name: arg.initial_weight for arg in bag.arguments.values()}
            
                for iteration in range(1, N+1):
            
                    # compute gradient for all arguments
                    gradient = compute_gradient(h, bag, preferred_order)
                    if all(value == 0 for value in gradient.values()): break
            
                    # update Adam state and update base scores
                    for arg in bag.arguments.values():
                        if arg.name not in immutable_args:
                            if arg.name not in m:
                                m[arg.name] = 0
                                v[arg.name] = 0
            
                            current_weight = arg.get_initial_weight()
                            adam_update, m[arg.name], v[arg.name] = adam_gradient(arg.name, gradient, m[arg.name], v[arg.name], iteration)
                            new_weight = current_weight - adam_update
                            new_weight = max(0, min(1, new_weight))
                            arg.reset_initial_weight(new_weight)
            
                    # recompute the strength and penalty
                    grad.algorithms.computeStrengthValues(bag, agg_f, inf_f)
            
                    if is_valid_order(bag, preferred_order):
                        break
            
                if is_valid_order(bag, preferred_order):
                    valid[i] = 1
                    diff = 0
                    for arg in bag.arguments.values():
                        diff += abs(arg.initial_weight - original_base_scores[arg.name])
                    if mutable_num:
                        base_score_diff[i] = diff/mutable_num
                predicted_strengths = [bag.arguments[name].strength for name in preferred_order]
                kendall[i], spearman[i] = compute_kendall_spearman(predicted_strengths)
                time_total[i] = time.time()-start_time
                
            print(f"### Experiments for {semantics}, {mutable_args}, {layer_sizes} ###")
            print(f"valid_itself:{valid_itself}")
            print(f"valid_avg:{(sum(valid)-valid_itself)/(N-valid_itself)}")
            print(f"kendall_avg:{(sum(kendall)-valid_itself)/(N-valid_itself)}")
            print(f"spearman_avg:{(sum(spearman)-valid_itself)/(N-valid_itself)}")
            print(f"runtime_avg:{sum(time_total)/(N-valid_itself)}")
            if sum(valid)-valid_itself == 0:
                diff = 0
            else:
                diff = sum(base_score_diff)/(sum(valid)-valid_itself)
            print(f"base_score_diff_avg:{diff}")